# Lire l'Explorateur Haïti par son API
*Reading the Haiti Explorer through its API*

Ce carnet lit l'API publique de l'Explorateur Haïti : gratuite, sans clé, sans inscription. Chaque valeur porte sa source, son année et son statut.

*This notebook reads the Haiti Explorer public API: free, keyless, no sign-up. Every value carries its source, year and status.*

Documentation : https://explorateur.atmart.ltd/api/index.html

## 1. Lire un fichier de l'API · *Reading an API file*

On s'identifie avec un en-tête `User-Agent` : le pare-feu du site refuse l'identifiant par défaut de `urllib` (erreur 403).

*Identify your program with a `User-Agent` header: the site's firewall refuses urllib's default identifier (error 403).*

In [ ]:
import json
import urllib.request

BASE = "https://explorateur.atmart.ltd/api/v1/"


def lire(chemin):
    """Lit un fichier JSON de l'API. / Reads one API JSON file."""
    req = urllib.request.Request(BASE + chemin, headers={"User-Agent": "carnet-explorateur/1.0"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.load(r)


index = lire("index.json")
print(index["version"], index["compte"])

## 2. Les territoires · *Territories*

Niveau 1 = département, 2 = arrondissement, 3 = commune. Chaque territoire connaît son parent.

*Level 1 = department, 2 = arrondissement, 3 = commune. Each territory knows its parent.*

In [ ]:
territoires = lire("territoires.json")["territoires"]
par_id = {t["id"]: t for t in territoires}
communes = [t for t in territoires if t["niveau"] == 3]
print(len(communes), "communes")


def departement(commune):
    """Remonte commune -> arrondissement -> département."""
    arr = par_id[commune["parent"]]
    return par_id[arr["parent"]]["nom_fr"]


print(communes[0]["nom_fr"], "->", departement(communes[0]))

## 3. Un indicateur, commune par commune · *One indicator, commune by commune*

Temps de trajet routier médian jusqu'à l'hôpital le plus proche (IND-ACC-002). Une valeur absente vaut `None`, jamais zéro.

*Median road travel time to the nearest hospital (IND-ACC-002). A missing value is `None`, never zero.*

In [ ]:
valeurs = lire("indicateurs/IND-ACC-002.json")["valeurs"]
connues = [v for v in valeurs if v["valeur"] is not None]
print(len(connues), "valeurs sur", len(valeurs))

for v in sorted(connues, key=lambda v: v["valeur"], reverse=True)[:5]:
    print(par_id[v["territoire"]]["nom_fr"], v["valeur"], v["unite"], v["annee"])

## 4. Un résumé par département · *A summary by department*

La médiane des communes de chaque département. Pour un temps d'accès, la médiane dit mieux qu'une moyenne la situation d'une commune ordinaire.

*The median of each department's communes: for a travel time, it describes an ordinary commune better than a mean.*

In [ ]:
import statistics
from collections import defaultdict

par_dep = defaultdict(list)
for v in connues:
    par_dep[departement(par_id[v["territoire"]])].append(v["valeur"])

for dep, vs in sorted(par_dep.items(), key=lambda kv: -statistics.median(kv[1])):
    print(f"{dep:<14} {statistics.median(vs):5.1f} min  ({len(vs)} communes)")

## 5. Avec pandas (facultatif) · *With pandas (optional)*

`pip install pandas`

In [ ]:
try:
    import pandas as pd
except ImportError:
    pd = None
    print("pandas n'est pas installé "
          "— pip install pandas")

if pd is not None:
    df = pd.DataFrame(connues)
    df["commune"] = df["territoire"].map(lambda i: par_id[i]["nom_fr"])
    df["departement"] = df["territoire"].map(lambda i: departement(par_id[i]))
    print(df.groupby("departement")["valeur"].describe().round(1))

## 6. Ce que l'indicateur ne dit pas · *What the indicator does not say*

Chaque indicateur porte ses limites, en quatre langues. Citez-les avec le chiffre, et citez la source de chaque valeur.

*Every indicator carries its limits in four languages. Quote them with the figure, and cite each value's source.*

In [ ]:
fiche = next(i for i in lire("indicateurs.json")["indicateurs"] if i["id"] == "IND-ACC-002")
print(fiche["noms"]["fr"], "/", fiche["noms"]["ht"])
print(fiche["limites"]["fr"])
print("Source :", connues[0]["source"])